# Level2 JAX PPO Full-DR Training Notebook

This notebook follows the same workflow as the JAX PPO training notebook, but uses `lsy_drone_racing.control.jax_ppo_fast_full_dr` so the train-only DR sections in `level2_dr.toml` are applied inside the pure-JAX rollout scan.

Recommended flow:

1. Check kernel and repository path.
2. Inspect the full-DR TOML settings loaded by the JAX path.
3. Run a small smoke train/eval.
4. Benchmark full-DR JAX rollout speed.
5. Run the full Level2 Fast2 validation workflow.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.55")

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo:", ROOT)
print("python:", sys.executable)
print("XLA preallocate:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("XLA memory fraction:", os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"))


Use the GPU pixi kernel for real training:

```text
.pixi/envs/gpu/bin/python
```

Install it if needed:

```bash
pixi run -e gpu python -m ipykernel install --user --name lsy-drone-racing-gpu --display-name "LSY Drone Racing (gpu)"
```


In [ ]:
%reload_ext autoreload
%autoreload 2

import json
from dataclasses import asdict

import jax
import optax

from lsy_drone_racing.control import jax_ppo, jax_ppo_fast_full_dr
from lsy_drone_racing.control.jax_ppo import load_jax_checkpoint
from lsy_drone_racing.control.jax_ppo_fast_full_dr import (
    benchmark_fast_rollout_full_dr,
    evaluate_ppo_fast_full_dr,
    level2_validation_args,
    load_full_dr_config,
    train_ppo_fast_full_dr,
)

print("jax:", jax.__version__)
print("devices:", jax.devices())
print("optax:", optax.__version__)
print("checkpoint format:", jax_ppo.CHECKPOINT_FORMAT)
print("full DR module:", jax_ppo_fast_full_dr.__name__)


## Settings


In [ ]:
CONFIG = "level2_dr.toml"
RUN_NAME = "jax_fast_full_dr_level2_fast2_validation"

CONFIG_PATH = ROOT / "config" / CONFIG
if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)

CHECKPOINT_DIR = ROOT / f"lsy_drone_racing/control/checkpoints/{RUN_NAME}"
MODEL_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_final.pkl"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

JAX_DEVICE = "gpu"
WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None
WANDB_MODE = "online"

NUM_ENVS_TRAIN = 1024
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 200_000_000
CHECKPOINT_INTERVAL_STEPS = 5_000_000
SUCCESS_THRESHOLD = 0.79

def build_level2_args(**overrides):
    base = dict(
        config=CONFIG,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        wandb_mode=WANDB_MODE,
        num_envs=NUM_ENVS_TRAIN,
        num_steps=NUM_STEPS,
    )
    base.update(overrides)
    return level2_validation_args(**base)

MODEL_PATH


## 1. Inspect Full-DR Config

This reads the same TOML file the trainer will use and shows the train-only DR values that are now carried in the JAX scan state.


In [ ]:
inspect_args = build_level2_args(jax_device="cpu", num_envs=8, total_timesteps=64)
full_dr_config = load_full_dr_config(inspect_args)
print(json.dumps(full_dr_config._asdict(), indent=2))


## 2. Smoke Train And Eval

This only checks that full-DR rollout, PPO update, checkpoint save/load, and eval all run. It is not a success-rate test.


In [ ]:
smoke_args = build_level2_args(
    jax_device="cpu",
    num_envs=8,
    num_steps=4,
    num_minibatches=2,
    update_epochs=1,
    total_timesteps=64,
    hidden_dim=32,
)

smoke_path = Path("/tmp/jax_ppo_full_dr_level2_smoke.pkl")
train_ppo_fast_full_dr(smoke_args, smoke_path, wandb_enabled=False)
load_jax_checkpoint(smoke_path)
evaluate_ppo_fast_full_dr(smoke_args, n_eval=2, model_path=smoke_path, seed_start=1)


## 3. Benchmark Full-DR JAX Rollout


In [ ]:
bench_args = build_level2_args(
    jax_device=JAX_DEVICE,
    num_envs=1024,
    num_steps=32,
    total_timesteps=1024 * 32,
    hidden_dim=64,
)
benchmark_fast_rollout_full_dr(bench_args, repeat=2, warmup=1)


## 4. Full Level2 Fast2 Validation

This is the full run used to check whether the full-DR JAX architecture can reproduce the accepted Level2 Fast2 success rate.


In [ ]:
train_args = build_level2_args(
    total_timesteps=TOTAL_TIMESTEPS_TRAIN,
    wandb_run_name=RUN_NAME,
    wandb_run_id=RUN_NAME,
    log_interval=20,
)

train_ppo_fast_full_dr(
    train_args,
    MODEL_PATH,
    wandb_enabled=WANDB_ENABLED,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_interval=CHECKPOINT_INTERVAL_STEPS,
)
print("saved:", MODEL_PATH)


In [ ]:
summary = evaluate_ppo_fast_full_dr(
    train_args,
    n_eval=100,
    model_path=MODEL_PATH,
    seed_start=1,
)
print(json.dumps({k: v for k, v in summary.items() if k != "episodes_detail"}, indent=2))
assert summary["success_rate"] >= SUCCESS_THRESHOLD


## 5. Equivalent Terminal Command


In [ ]:
print(
    "pixi run -e gpu python scripts/validate_level2_jax_ppo_fast_full_dr.py "
    "--eval-episodes 100 --eval-seed-start 1 --success-threshold 0.79"
)
